In [ ]:
import subprocess
subprocess.run([
    "pip", "install",
    "numpy==2.2.4",
    "pandas==2.2.2",
    "--force-reinstall"
], check=True)

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install facenet-pytorch==2.6.0 moviepy --quiet
!pip install decord --quiet   # fast video reader fallback (optional)

In [ ]:
# ── Cell 2: Imports & version check ──────────────────────────────────────────
import os, gc, time, pickle, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from facenet_pytorch import MTCNN
from moviepy.editor import VideoFileClip
from PIL import Image
import torchvision
import torchvision.transforms as T
from torchvision.models.video import r3d_18, R3D_18_Weights

warnings.filterwarnings("ignore")

print(f"torch        : {torch.__version__}")
print(f"torchvision  : {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
# ─── Paths ───
BASE            = "/kaggle/input/datasets/animesh2cool/first-impressions-v2-cvpr17-training"
ANNOTATION_PATH = "/kaggle/input/datasets/animesh2cool/first-impressions-v2-cvpr17-training/train-annotation/annotation_training.pkl"
SAVE_DIR        = "/kaggle/working/checkpoints"
CACHE_DIR       = "/kaggle/working/face_cache"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ─── Video / model ───
NUM_FRAMES   = 16      # frames sampled uniformly per clip
IMG_SIZE     = 112     # R3D-18 default spatial size
BATCH_SIZE   = 4
CHUNK_SIZE   = 500
NUM_EPOCHS   = 30
LR           = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 5       # early-stopping patience (chunks with no val improvement)
DROPOUT      = 0.5     # dropout rate in model heads (increased from 0.4)

# ─── Split sizes ───
VAL_SIZE  = 0.08       # 8 % validation
TEST_SIZE = 0.08       # 8 % held-out test (never seen during training)

# ─── Big Five trait labels ───
TRAITS     = ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]
TRAIT_ABBR = ["open",    "consc",             "extra",        "agree",          "neuro"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Config : {NUM_FRAMES} frames | {IMG_SIZE}px | batch {BATCH_SIZE} | chunk {CHUNK_SIZE}")
print(f"Splits : val={VAL_SIZE*100:.0f}%  test={TEST_SIZE*100:.0f}%  train=rest")


In [ ]:
# ── Cell 4: Load annotations & map video paths ────────────────────────────────
with open(ANNOTATION_PATH, "rb") as f:
    annotations = pickle.load(f, encoding="latin1")

rows = {}
for trait in TRAITS:
    for video, score in annotations[trait].items():
        if video not in rows:
            rows[video] = {"video": video}
        rows[video][trait] = float(score)

df = pd.DataFrame(rows.values())

# ── map filenames → absolute paths ──
video_paths = {}
for split in os.listdir(BASE):
    if split.startswith("train-") and split != "train-annotation":
        split_path = os.path.join(BASE, split)
        for folder in os.listdir(split_path):
            folder_path = os.path.join(split_path, folder)
            if not os.path.isdir(folder_path):
                continue
            for fname in os.listdir(folder_path):
                if fname.endswith(".mp4"):
                    video_paths[fname] = os.path.join(folder_path, fname)

df["path"] = df["video"].map(video_paths)
df = df.dropna(subset=["path"]).reset_index(drop=True)
print(f"Total valid samples: {len(df)}")
print(df[TRAITS].describe().round(4))

In [ ]:
# ── Cell 5: MTCNN shared instance ─────────────────────────────────────────────
# Created ONCE here — passed into the dataset so workers never re-load weights.
# keep_all=False → return only the highest-confidence face per frame.
mtcnn_detector = MTCNN(
    image_size=IMG_SIZE,
    margin=20,
    min_face_size=40,
    keep_all=False,
    post_process=True,   # normalise to [-1, 1]
    device=DEVICE,
)
print("MTCNN ready.")

In [ ]:
# ── Cell 6: Video preprocessing helpers ───────────────────────────────────────
def preprocess_video_to_cache(video_path: str, num_frames: int = NUM_FRAMES,
                               img_size: int = IMG_SIZE) -> str:
    """
    Decode a video with MoviePy, run MTCNN on each sampled frame,
    save the face tensor to disk, and return the cache path.

    Memory discipline:
      - VideoFileClip is closed immediately after frame extraction.
      - PIL images are deleted frame-by-frame.
      - MTCNN runs on CPU-converted frames to avoid accumulating GPU tensors.
    """
    cache_key  = os.path.splitext(os.path.basename(video_path))[0] + ".pt"
    cache_path = os.path.join(CACHE_DIR, cache_key)

    if os.path.exists(cache_path):
        return cache_path   # already processed

    blank = torch.zeros(3, img_size, img_size)
    face_tensors = []

    try:
        clip     = VideoFileClip(video_path)
        duration = clip.duration
        # Cap timestamps so we never request a frame past the last decodeable frame
        ts_end   = max(0.0, duration - 0.05)
        timestamps = np.linspace(0.0, ts_end, num_frames)

        for t in timestamps:
            try:
                frame_np = clip.get_frame(float(t))           # H×W×3 uint8 RGB
                pil_img  = Image.fromarray(frame_np.astype(np.uint8))
                face     = mtcnn_detector(pil_img)             # tensor [3,H,W] or None
                del pil_img, frame_np
            except Exception:
                face = None

            face_tensors.append(face if face is not None else blank.clone())

        clip.reader.close()
        if hasattr(clip, 'audio') and clip.audio is not None:
            clip.audio.reader.close_proc()
        clip.close()
        del clip

    except Exception as e:
        print(f"  [WARN] Could not decode {os.path.basename(video_path)}: {e}")
        face_tensors = [blank.clone() for _ in range(num_frames)]

    stacked = torch.stack(face_tensors)   # (T, 3, H, W)
    torch.save(stacked, cache_path)

    del face_tensors, stacked
    gc.collect()
    return cache_path


def warm_cache_for_chunk(paths, verbose=True):
    """Pre-process a list of video paths sequentially (one at a time)."""
    t0 = time.time()
    cached, errors = 0, 0
    n = len(paths)
    for i, p in enumerate(paths):
        try:
            preprocess_video_to_cache(p)
            cached += 1
        except Exception as e:
            errors += 1
            if verbose:
                print(f"  [ERROR] {os.path.basename(p)}: {e}")
        if verbose and (i + 1) % 100 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta  = (n - i - 1) / rate
            print(f"  Cached {i+1}/{n} | {rate:.1f} vid/s | ETA {eta/60:.1f} min")

    if verbose:
        print(f"  Done: {cached} cached, {errors} errors in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Cell 7: Dataset (with optional augmentation) ──────────────────────────────
# Augmentation applied PER FRAME on the temporal dimension during training.
# At inference / validation / test  →  augment=False (only normalisation).

_train_augment = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3,
                                  saturation=0.2, hue=0.05)], p=0.5),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))],   p=0.2),
    T.RandomGrayscale(p=0.05),
])

class PersonalityDataset(Dataset):
    """
    Loads pre-cached face tensors from disk.
    augment=True  → applies spatial/colour augmentation per frame (training only).
    """
    def __init__(self, df, augment: bool = False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row        = self.df.iloc[idx]
        cache_key  = os.path.splitext(os.path.basename(row["path"]))[0] + ".pt"
        cache_path = os.path.join(CACHE_DIR, cache_key)

        try:
            faces = torch.load(cache_path, map_location="cpu")  # (T, 3, H, W)
        except Exception:
            faces = torch.zeros(NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE)

        # R3D expects (C, T, H, W)
        faces = faces.permute(1, 0, 2, 3).float()

        # Rescale MTCNN output [-1,1] → [0,1] then ImageNet-normalise
        faces = (faces + 1.0) / 2.0

        # ── Augment each frame independently (training only) ─────────────────
        if self.augment:
            # faces: (C, T, H, W) → apply transform on each (C, H, W) slice
            aug_frames = []
            for t in range(faces.shape[1]):
                frame = faces[:, t, :, :]          # (C, H, W) in [0,1]
                frame = _train_augment(frame)       # T.* work on [0,1] tensors
                aug_frames.append(frame)
            faces = torch.stack(aug_frames, dim=1)  # (C, T, H, W)

        mean  = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
        std   = torch.tensor([0.22803, 0.22145,  0.216989]).view(3, 1, 1, 1)
        faces = (faces - mean) / std

        labels = torch.tensor([float(row[t]) for t in TRAITS], dtype=torch.float32)
        return faces, labels


In [ ]:
# ── Cell 8: R3D-18 model (with configurable Dropout) ─────────────────────────
class PersonalityR3D(nn.Module):
    """
    R3D-18 backbone (pretrained Kinetics-400) with five independent
    regression heads — one per Big-Five personality trait.

    Dropout is applied in both the shared projection and per-trait heads
    to regularise against overfitting (rate controlled by DROPOUT config).
    """
    def __init__(self, num_traits: int = 5, dropout: float = DROPOUT):
        super().__init__()

        backbone = r3d_18(weights=R3D_18_Weights.KINETICS400_V1)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        in_features = 512

        # ── Shared projection with stronger dropout ──
        self.shared_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),            # <── regularisation
        )

        # ── Per-trait heads ──
        self.trait_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(256, 64),
                nn.GELU(),
                nn.Dropout(dropout / 2),    # <── regularisation
                nn.Linear(64, 1),
                nn.Sigmoid(),
            )
            for _ in range(num_traits)
        ])

    def forward(self, x):
        feats  = self.feature_extractor(x)
        shared = self.shared_head(feats)
        preds  = [head(shared) for head in self.trait_heads]
        return torch.cat(preds, dim=1)


# Sanity-check
_model = PersonalityR3D().to(DEVICE)
_dummy = torch.zeros(2, 3, NUM_FRAMES, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    _out = _model(_dummy)
print(f"R3D output shape: {_out.shape}   ← should be (2, 5)")
print(f"Dropout rate    : {DROPOUT} (shared)  {DROPOUT/2:.2f} (per-trait)")
del _dummy, _out
torch.cuda.empty_cache()


In [ ]:
# ── Cell 9: Evaluation & benchmark utilities ───────────────────────────────────
def evaluate(model, loader, criterion, device=DEVICE):
    """Return overall 1-error, MSE, per-trait 1-error and per-trait MSE."""
    model.eval()
    total_loss  = 0.0
    trait_mses  = torch.zeros(len(TRAITS), device=device)
    n_batches   = 0

    with torch.no_grad():
        for videos, labels in loader:
            videos, labels = videos.to(device), labels.to(device)
            with autocast():
                outputs = model(videos)
                loss    = criterion(outputs, labels)
            total_loss += loss.item()
            trait_mses += torch.mean((outputs - labels) ** 2, dim=0)
            n_batches  += 1

    if n_batches == 0:
        return 0.0, 0.0, np.zeros(len(TRAITS)), np.zeros(len(TRAITS))

    avg_mse        = total_loss / n_batches
    trait_mses_avg = (trait_mses / n_batches).cpu().numpy()
    trait_1errors  = 1.0 - trait_mses_avg
    overall_1error = float(1.0 - avg_mse)
    return overall_1error, avg_mse, trait_1errors, trait_mses_avg


def print_metrics(overall_1err, overall_mse, trait_1errs, trait_mses, header=""):
    if header:
        print(f"  {header}")
    print(f"  {'Trait':20s}  {'MSE':>8}  {'1-Error':>8}")
    print(f"  {'─'*20}  {'─'*8}  {'─'*8}")
    for name, mse, one_err in zip(TRAITS, trait_mses, trait_1errs):
        print(f"  {name:20s}  {mse:8.4f}  {one_err:8.4f}")
    print(f"  {'─'*20}  {'─'*8}  {'─'*8}")
    print(f"  {'OVERALL':20s}  {overall_mse:8.4f}  {overall_1err:8.4f}")


class Benchmark:
    """Accumulates per-epoch wall-clock & throughput metrics."""
    def __init__(self):
        self.records = []

    def log(self, chunk, epoch, train_loss, val_mse, one_err,
            epoch_time_s, n_samples):
        self.records.append({
            "chunk":       chunk,
            "epoch":       epoch,
            "train_loss":  round(train_loss, 5),
            "val_mse":     round(val_mse, 5),
            "1_error":     round(one_err, 5),
            "epoch_s":     round(epoch_time_s, 2),
            "vid_per_s":   round(n_samples / epoch_time_s, 2),
        })

    def summary_df(self):
        return pd.DataFrame(self.records)

    def save(self, path):
        self.summary_df().to_csv(path, index=False)
        print(f"  Benchmark saved → {path}")

In [ ]:
# ── Cell 10: Train / Validation / Test split ──────────────────────────────────
#
#   Full dataset  →  8 % TEST  (held-out, never seen during training)
#                 →  8 % VAL   (used for early-stopping & best-model selection)
#                 →  84% TRAIN
#
# The test set is intentionally set aside here and NOT touched until Cell 15.

# Step 1: carve out the test set
train_val_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=42)

# Step 2: carve out the validation set from the remainder
# Adjust val fraction so it is 8 % of the TOTAL (not 8 % of the 92 % remainder)
adjusted_val = VAL_SIZE / (1.0 - TEST_SIZE)
train_df, val_df = train_test_split(train_val_df, test_size=adjusted_val, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

n_total = len(df)
print(f"Total   : {n_total}")
print(f"Train   : {len(train_df)}  ({len(train_df)/n_total*100:.1f} %)")
print(f"Val     : {len(val_df)}   ({len(val_df)/n_total*100:.1f} %)")
print(f"Test    : {len(test_df)}  ({len(test_df)/n_total*100:.1f} %)  ← held-out")

# ── Pre-cache validation set (done once) ──
print("\nPre-caching validation videos...")
warm_cache_for_chunk(val_df["path"].tolist(), verbose=True)

val_ds     = PersonalityDataset(val_df, augment=False)   # no augmentation on val
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print("\nPre-caching test videos (done once, used only in Cell 15)...")
warm_cache_for_chunk(test_df["path"].tolist(), verbose=True)
print("Test set cached and ready.")


In [ ]:
# ── Cell 11: Training loop (chunk-based, with epoch-level early stopping) ────
#
# EARLY STOPPING:
#   - Tracks best val MSE across ALL epochs (not just per-chunk).
#   - Triggers when val MSE has not improved for PATIENCE consecutive CHUNKS.
#   - Best model weights saved to BEST_PATH whenever val MSE improves.
#
# AUGMENTATION:
#   - Training DataLoader uses PersonalityDataset(augment=True).
#   - Val / Test DataLoaders always use augment=False.

import json as _json

PROGRESS_FILE = os.path.join(SAVE_DIR, "progress.json")
BEST_PATH     = os.path.join(SAVE_DIR, "personality_r3d_best.pth")

def _save_progress(last_chunk, best_mse, no_improve, elapsed):
    with open(PROGRESS_FILE, "w") as _f:
        _json.dump({
            "last_completed_chunk": last_chunk,
            "best_val_mse":         best_mse,
            "no_improve_cnt":       no_improve,
            "global_elapsed_s":     elapsed,
        }, _f, indent=2)

def _load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as _f:
            return _json.load(_f)
    return None

progress   = _load_progress()
num_chunks = (len(train_df) + CHUNK_SIZE - 1) // CHUNK_SIZE

if progress and progress["last_completed_chunk"] > 0:
    resume_from_chunk = progress["last_completed_chunk"]
    best_val_mse      = progress["best_val_mse"]
    no_improve_cnt    = progress["no_improve_cnt"]
    prev_elapsed      = progress["global_elapsed_s"]

    resume_ckpt_path = os.path.join(
        SAVE_DIR, f"personality_r3d_chunk_{resume_from_chunk:03d}.pth"
    )
    print(f"\n{'='*60}")
    print(f"  ▶ RESUMING from chunk {resume_from_chunk}/{num_chunks}")
    print(f"    Checkpoint : {resume_ckpt_path}")
    print(f"    Best ValMSE: {best_val_mse:.4f}  |  No-improve: {no_improve_cnt}/{PATIENCE}")
    print(f"    Prev elapsed: {prev_elapsed/60:.1f} min")
    print(f"{'='*60}\n")

    ckpt = torch.load(resume_ckpt_path, map_location=DEVICE)
    model     = PersonalityR3D().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])

    criterion = nn.MSELoss()
    scaler    = GradScaler()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS * num_chunks
    )
    steps_done = resume_from_chunk * NUM_EPOCHS
    for _ in range(steps_done):
        scheduler.step()

    benchmark       = Benchmark()
    start_chunk_idx = resume_from_chunk
    global_start    = time.time() - prev_elapsed

else:
    print(f"\n{'='*60}")
    print(f"Training R3D-18 | {num_chunks} chunks × {NUM_EPOCHS} epochs  [fresh start]")
    print(f"Dropout={DROPOUT}  |  EarlyStopping patience={PATIENCE} chunks")
    print(f"Augmentation: RandomHFlip + ColorJitter + GaussianBlur + Grayscale")
    print(f"{'='*60}")

    model     = PersonalityR3D().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()
    scaler    = GradScaler()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS * num_chunks
    )
    benchmark       = Benchmark()
    best_val_mse    = float("inf")
    no_improve_cnt  = 0
    start_chunk_idx = 0
    global_start    = time.time()

# ── Main training loop ────────────────────────────────────────────────────────
for chunk_idx in range(start_chunk_idx, num_chunks):
    chunk_start = chunk_idx * CHUNK_SIZE
    chunk_end   = min(chunk_start + CHUNK_SIZE, len(train_df))
    chunk_df    = train_df.iloc[chunk_start:chunk_end]

    print(f"\n{'─'*60}")
    print(f"CHUNK {chunk_idx+1}/{num_chunks}  "          f"(videos {chunk_start}–{chunk_end-1}, n={len(chunk_df)})")
    print(f"{'─'*60}")

    # ── Step 1: Pre-cache chunk ───────────────────────────────────────────────
    print("  [1/3] Caching faces with MoviePy + MTCNN...")
    cache_t0 = time.time()
    warm_cache_for_chunk(chunk_df["path"].tolist(), verbose=True)
    print(f"  Cache time: {(time.time()-cache_t0)/60:.1f} min")
    gc.collect(); torch.cuda.empty_cache()

    # ── Step 2: Build DataLoader with augmentation ────────────────────────────
    chunk_ds     = PersonalityDataset(chunk_df, augment=True)   # ← augment ON
    chunk_loader = DataLoader(chunk_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    n_train_samples = len(chunk_df)

    # ── Step 3: Epoch loop ────────────────────────────────────────────────────
    print(f"  [2/3] Training {NUM_EPOCHS} epoch(s) with augmentation...")
    last_val_mse = None

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        epoch_loss  = 0.0
        epoch_start = time.time()
        n_batches   = 0

        for videos, labels in chunk_loader:
            videos, labels = videos.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            with autocast():
                outputs = model(videos)
                loss    = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            n_batches  += 1

        scheduler.step()

        epoch_time     = time.time() - epoch_start
        avg_train_loss = epoch_loss / max(n_batches, 1)
        overall_1err, val_mse, t_1errs, t_mses = evaluate(model, val_loader, criterion)
        last_val_mse = val_mse

        benchmark.log(
            chunk=chunk_idx + 1, epoch=epoch,
            train_loss=avg_train_loss, val_mse=val_mse, one_err=overall_1err,
            epoch_time_s=epoch_time, n_samples=n_train_samples,
        )

        print(
            f"  Epoch [{epoch:02d}/{NUM_EPOCHS}]  "            f"TrainLoss: {avg_train_loss:.4f}  "            f"ValMSE: {val_mse:.4f}  "            f"1-Error: {overall_1err:.4f}  "            f"LR: {scheduler.get_last_lr()[0]:.2e}  "            f"Time: {epoch_time:.1f}s  "            f"({n_train_samples/epoch_time:.1f} vid/s)"
        )
        print_metrics(overall_1err, val_mse, t_1errs, t_mses)
        torch.cuda.empty_cache(); gc.collect()

    # ── Save chunk checkpoint ─────────────────────────────────────────────────
    ckpt_path = os.path.join(SAVE_DIR, f"personality_r3d_chunk_{chunk_idx+1:03d}.pth")
    torch.save({
        "chunk":           chunk_idx + 1,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "val_mse":         last_val_mse,
        "config": {"NUM_FRAMES": NUM_FRAMES, "IMG_SIZE": IMG_SIZE, "TRAITS": TRAITS},
    }, ckpt_path)
    print(f"  [3/3] Chunk checkpoint → {ckpt_path}")

    # ── Early stopping check ──────────────────────────────────────────────────
    if last_val_mse < best_val_mse:
        best_val_mse   = last_val_mse
        no_improve_cnt = 0
        torch.save({
            "chunk":       chunk_idx + 1,
            "model_state": model.state_dict(),
            "val_mse":     best_val_mse,
            "config": {"NUM_FRAMES": NUM_FRAMES, "IMG_SIZE": IMG_SIZE, "TRAITS": TRAITS},
        }, BEST_PATH)
        print(f"  ✅ New best model saved (ValMSE={best_val_mse:.4f})")
    else:
        no_improve_cnt += 1
        print(f"  ⚠️  No val improvement: {no_improve_cnt}/{PATIENCE}")
        if no_improve_cnt >= PATIENCE:
            print(f"  ⛔ Early stopping triggered after {chunk_idx+1} chunks.")
            break

    # ── Save resume progress ──────────────────────────────────────────────────
    elapsed_so_far = time.time() - global_start
    _save_progress(chunk_idx + 1, best_val_mse, no_improve_cnt, elapsed_so_far)
    print(f"  💾 Progress saved  (elapsed {elapsed_so_far/60:.1f} min)")

    del chunk_ds, chunk_loader
    gc.collect(); torch.cuda.empty_cache()
    benchmark.save(os.path.join(SAVE_DIR, "benchmark.csv"))

total_time = (time.time() - global_start) / 60
print(f"\n{'='*60}")
print(f"Training complete in {total_time:.1f} min")
print(f"Best Val MSE : {best_val_mse:.4f}")
print(f"Best model   : {BEST_PATH}")
print(f"{'='*60}")
print("\n→ Run Cell 15 to evaluate on the held-out TEST set.")


In [ ]:
import torch, json, os

# Check progress file
p = "/kaggle/working/checkpoints/progress.json"
if os.path.exists(p):
    with open(p) as f:
        progress = json.load(f)
    print("progress.json found:")
    print(json.dumps(progress, indent=2))
else:
    print("progress.json NOT found")

# Check which chunk checkpoints exist
ckpt_dir = "/kaggle/working/checkpoints"
ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.startswith("personality_r3d_chunk")])
print(f"\nChunk checkpoints found ({len(ckpts)}):")
for c in ckpts:
    path = os.path.join(ckpt_dir, c)
    size = os.path.getsize(path) / 1e6
    ckpt = torch.load(path, map_location="cpu")
    print(f"  {c}  ({size:.1f} MB)  |  chunk={ckpt['chunk']}  val_mse={ckpt['val_mse']:.4f}")

# Load and inspect the latest chunk checkpoint
if ckpts:
    latest = os.path.join(ckpt_dir, ckpts[-1])
    ckpt = torch.load(latest, map_location="cpu")
    print(f"\nLatest checkpoint: {ckpts[-1]}")
    print(f"  Chunk     : {ckpt['chunk']}")
    print(f"  Val MSE   : {ckpt['val_mse']:.4f}")
    print(f"  Config    : {ckpt['config']}")

In [ ]:
# ── Cell 12: Final benchmark summary ──────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")   # headless on Kaggle
import matplotlib.pyplot as plt

bm_df = benchmark.summary_df()
print("\n── Benchmark Summary ──")
print(bm_df.groupby("chunk")[["train_loss","val_mse","1_error",
                               "epoch_s","vid_per_s"]].mean().round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(bm_df["train_loss"].values, label="Train Loss")
axes[0].plot(bm_df["val_mse"].values,   label="Val MSE")
axes[0].set_title("Loss curves")
axes[0].set_xlabel("Global epoch"); axes[0].legend()

axes[1].plot(bm_df["1_error"].values, color="green")
axes[1].set_title("Overall 1-Error (higher = better)")
axes[1].set_xlabel("Global epoch")

axes[2].plot(bm_df["vid_per_s"].values, color="orange")
axes[2].set_title("Throughput (videos / sec)")
axes[2].set_xlabel("Global epoch")

plt.tight_layout()
fig_path = os.path.join(SAVE_DIR, "training_curves.png")
plt.savefig(fig_path, dpi=120)
print(f"\nPlot saved → {fig_path}")
plt.show()

In [ ]:
# ── Cell 13: Download weights from Kaggle output tab ──────────────────────────
import shutil
from IPython.display import FileLink, display

WORKING = "/kaggle/working"

print("\n── Saved checkpoints ──")
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname}  ({size:.1f} MB)")

# Copy best model & benchmark to /kaggle/working (Output tab)
best_dst  = os.path.join(WORKING, "personality_r3d_best.pth")
bm_dst    = os.path.join(WORKING, "benchmark.csv")
curve_dst = os.path.join(WORKING, "training_curves.png")

for src, dst in [
    (BEST_PATH,   best_dst),
    (os.path.join(SAVE_DIR, "benchmark.csv"),       bm_dst),
    (os.path.join(SAVE_DIR, "training_curves.png"), curve_dst),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied → {dst}")
        display(FileLink(dst))

print("\n→ Use the Kaggle Output tab (right panel) to download all files.")

In [ ]:
# ── Cell 15: Test-set evaluation (held-out, run ONCE after training) ──────────
#
# Loads the BEST checkpoint and evaluates it on the 8 % test split
# that was set aside in Cell 10 and never seen during training.
#
# Metrics reported:
#   • Per-trait MSE and 1-Error
#   • Overall (mean) MSE and 1-Error
#   • Scatter plots: predicted vs ground-truth for each trait

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("Loading best checkpoint for test evaluation...")
ckpt_test = torch.load(BEST_PATH, map_location=DEVICE)
test_model = PersonalityR3D().to(DEVICE)
test_model.load_state_dict(ckpt_test["model_state"])
test_model.eval()
print(f"  Checkpoint from chunk {ckpt_test['chunk']}  |  ValMSE={ckpt_test['val_mse']:.4f}")

# Build test loader (NO augmentation — clean evaluation)
test_ds     = PersonalityDataset(test_df, augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

criterion_test = nn.MSELoss()
overall_1err, test_mse, t_1errs, t_mses = evaluate(test_model, test_loader,
                                                     criterion_test, device=DEVICE)

print("\n" + "="*55)
print("  HELD-OUT TEST SET RESULTS")
print("="*55)
print_metrics(overall_1err, test_mse, t_1errs, t_mses, header="Per-trait metrics")

# ── Collect all predictions + ground truth for scatter plots ─────────────────
all_preds  = []
all_labels = []
test_model.eval()
with torch.no_grad():
    for videos, labels in test_loader:
        videos = videos.to(DEVICE)
        with autocast():
            preds = test_model(videos)
        all_preds.append(preds.cpu())
        all_labels.append(labels)

all_preds  = torch.cat(all_preds,  dim=0).numpy()   # (N, 5)
all_labels = torch.cat(all_labels, dim=0).numpy()   # (N, 5)

# ── Scatter plots: predicted vs ground-truth ─────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (trait, ax) in enumerate(zip(TRAITS, axes)):
    ax.scatter(all_labels[:, i], all_preds[:, i], alpha=0.4, s=10, color="steelblue")
    lo = min(all_labels[:, i].min(), all_preds[:, i].min())
    hi = max(all_labels[:, i].max(), all_preds[:, i].max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1)      # perfect-prediction line
    ax.set_title(f"{trait}\nMSE={t_mses[i]:.4f}  1-Err={t_1errs[i]:.4f}")
    ax.set_xlabel("Ground truth")
    ax.set_ylabel("Predicted")
    ax.set_xlim(lo - 0.02, hi + 0.02)
    ax.set_ylim(lo - 0.02, hi + 0.02)
    ax.set_aspect("equal")

plt.suptitle(f"Test set  —  Overall MSE: {test_mse:.4f}  |  Overall 1-Error: {overall_1err:.4f}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
scatter_path = os.path.join(SAVE_DIR, "test_scatter.png")
plt.savefig(scatter_path, dpi=120)
print(f"\nScatter plots saved → {scatter_path}")
plt.show()

# ── MAE as additional metric ─────────────────────────────────────────────────
mae_per_trait = np.mean(np.abs(all_preds - all_labels), axis=0)
overall_mae   = float(np.mean(mae_per_trait))
print("\n── Mean Absolute Error (MAE) per trait ──")
for name, mae in zip(TRAITS, mae_per_trait):
    print(f"  {name:20s}  {mae:.4f}")
print(f"  {'OVERALL':20s}  {overall_mae:.4f}")


In [ ]:
# ── Cell 14: Inference helper (optional) ──────────────────────────────────────
def predict_single_video(video_path: str, weights_path: str = BEST_PATH):
    """
    Run personality prediction on a single video.
    Returns a dict mapping each trait → predicted score [0, 1].
    """
    model_inf = PersonalityR3D().to(DEVICE)
    ckpt      = torch.load(weights_path, map_location=DEVICE)
    model_inf.load_state_dict(ckpt["model_state"])
    model_inf.eval()

    cache_path = preprocess_video_to_cache(video_path)
    faces      = torch.load(cache_path, map_location="cpu")
    faces      = faces.permute(1, 0, 2, 3).float()
    faces      = (faces + 1.0) / 2.0
    mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
    std  = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)
    faces = (faces - mean) / std
    faces = faces.unsqueeze(0).to(DEVICE)  # (1, C, T, H, W)

    with torch.no_grad():
        preds = model_inf(faces).cpu().numpy()[0]

    result = {trait: float(score) for trait, score in zip(TRAITS, preds)}
    print("\nPredicted Big-Five scores:")
    for k, v in result.items():
        bar = '█' * int(v * 20)
        print(f"  {k:20s} {v:.3f}  |{bar}")
    return result

# Example (uncomment to test):
# predict_single_video("/path/to/video.mp4")